# Random Forest

Il Random Forest è un algoritmo di machine learning supervisionato basato su una collezionei alberi decisionali indipendenti, combinati per migliorare la accuratezza e la robustezza rispetto a singoli alberi.

In [13]:
import pandas as pd
from joblib import Parallel, delayed
import numpy as np
import math
from sklearn.model_selection import GroupKFold, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.multioutput import MultiOutputClassifier
from sklearn.metrics import f1_score, make_scorer, classification_report
import tqdm
from tabulate import tabulate
from pathlib import Path
import warnings
import time
from datetime import timedelta
import re
# Nascondo i warning
warnings.filterwarnings('ignore')

# Definisco il percorso dei file
FILE_PATH = Path('/Users/francesco/Tesi/BC-ML4/dataset/cleaned')

# Lista dei csv su cui fare training
datasets = {
    't2_medsam': FILE_PATH / 't2_medsam_masks.csv',
    't2_preprocessed': FILE_PATH / 't2_preprocessed_masks.csv',
    't2_original': FILE_PATH / 't2_original_masks.csv',
    'medsam_dynamic': FILE_PATH / 'medsam_dynamic.csv',
    'preprocessed_dynamic': FILE_PATH / 'preprocessed_dynamic.csv',
    'original_dynamic': FILE_PATH / 'original_dynamic.csv'
}

# Training

In [ ]:
def training(file_path, csv_name):

    # Prima cosa: leggo il file CSV che mi hai dato.
    df = pd.read_csv(file_path)

    # Definisco quali sono le colonne che voglio predire.
    original_target_list = ['PR [SII]', 'ER [SII]', 'KI67 [%]', 'HER2 [SII]']

    
    df_validi = df.dropna(subset=original_target_list).copy()


    df_validi['PR_class'] = (df_validi['PR [SII]'] > 0.5).astype(int)
    df_validi['ER_class'] = (df_validi['ER [SII]'] > 0.5).astype(int)
    df_validi['KI67_class'] = (df_validi['KI67 [%]'] >= 20).astype(int)
    df_validi['HER2_class'] = (df_validi['HER2 [SII]'] >= 3).astype(int)

    # Creo una lista con i nomi delle mie nuove colonne target binarie.
    final_target_list = ['PR_class', 'ER_class', 'KI67_class', 'HER2_class']

    
    features_to_drop = ['Patient ID', 'lesion idx', 'tumor/benign', 'GRADE', 'isTN', 'Breast'] + original_target_list + final_target_list
    features = df_validi.drop(columns=features_to_drop, errors='ignore')

    
    target = df_validi[final_target_list]
    groups = df_validi['Patient ID']

    # I valori mancanti li riempio con la media
    features = features.fillna(features.mean())

    # Vado a rimuvoere i carattei speciali
    features.columns = [re.sub(r'\[|\]|<', '', col) for col in features.columns]


    cv = GroupKFold(n_splits=5)

    base_model = RandomForestClassifier(random_state=42, n_jobs=1)
    multi_output_model = MultiOutputClassifier(base_model)

    # Definisco gli iperparametri da usare
    iperparametri = {
      'estimator__n_estimators': [50, 100, 150, 200],
      'estimator__max_depth': [3, 5, 7],
      'estimator__min_samples_leaf': [4, 6],
      'estimator__class_weight': ['balanced']
    }

     # Funzione per calcolare lo score medio su tutti i 4 target
    def multi_f1_scorer(y_true, y_pred):
        y_true = np.array(y_true)
        y_pred = np.array(y_pred)
        
        scores = []
        for i in range(y_true.shape[1]):
            # Calcolo l'F1 per ogni colonna separatamente
            s = f1_score(y_true[:, i], y_pred[:, i], average='macro', zero_division=0)
            scores.append(s)
            
        # Restituisco la media degli score. Questo numero singolo guida la GridSearch.
        return np.mean(scores)


    scorer = make_scorer(multi_f1_scorer)

    # Stampo quante combinazioni proverò
    total_combinations = math.prod(len(v) for v in iperparametri.values())
    print(f"\nInizio Grid Search ({total_combinations} combinazioni) per: {csv_name}")

    # Configuro la Grid Search CV 
    grid_search = GridSearchCV(
        estimator=multi_output_model,
        param_grid=iperparametri,
        cv=cv,
        scoring=scorer,
        n_jobs=-1,
        verbose=1,
        refit=True,
        error_score='raise'
    )

    grid_search.fit(features, target, groups=groups)

    # Mi salvo i risultati migliori
    best_params = grid_search.best_params_
    best_score = grid_search.best_score_


    fold_reports = []
    for train_idx, test_idx in cv.split(features, target, groups):
        # Divido i dati in train e test
        X_train, X_test = features.iloc[train_idx], features.iloc[test_idx]
        y_train, y_test = target.iloc[train_idx], target.iloc[test_idx]

        # Prendo il modello migliore e lo vado ad allenare nuovamente
        model_clone = grid_search.best_estimator_
        model_clone.fit(X_train, y_train)
        y_pred = model_clone.predict(X_test)

        # Genero il report di classificazione dettagliato per questo fold.
        report_dict = {}
        for i, col in enumerate(final_target_list):
            report_dict[col] = classification_report(
                y_test.iloc[:, i],
                y_pred[:, i],
                output_dict=True,
                zero_division=0
            )
        fold_reports.append(report_dict)

    final_result = [{
        **{k.replace('estimator__', ''): v for k, v in best_params.items()},
        'mean_score': best_score,
        'std_score': grid_search.cv_results_['std_test_score'][grid_search.best_index_],
        'fold_reports': fold_reports
    }]

    # E restituisco il risultato. Finito!
    return final_result


# Stampo i risultati in un formato piú leggibile

In [15]:
def print_grid_search_results(results_per_dataset):

    print("\n" + "=" * 80)
    print(" " * 25 + "RIEPILOGO DEI MIGLIORI RISULTATI")
    print("=" * 80)

    summary_data = []

    for name, metrics_list in results_per_dataset.items():
        best_result = metrics_list[0]

        print(f"\n{'─' * 80}")
        print(f" Dataset: {name}")
        print(f"{'─' * 80}")
        print(f"\n Performance: F1-score = {best_result['mean_score']:.3f} ± {best_result['std_score']:.3f}\n")

        print("Iperparametri Ottimali:")
        possible_params = [
            ('N Estimators', 'n_estimators'),
            ('Max Depth', 'max_depth'),
            ('Min Samples Leaf', 'min_samples_leaf'),
            ('Class Weight', 'class_weight'),
            ('Min Samples Split', 'min_samples_split'), 
            ('Max Features', 'max_features')            
        ]

        params_table = []
        for label, key in possible_params:
            if key in best_result:
                val = best_result[key]
                params_table.append([label, val])

        print(tabulate(params_table, headers=['Parametro', 'Valore'], tablefmt='simple'))

        print("\n Metriche di Classificazione per Target (Dettaglio):\n")
        target_names = ['PR_class', 'ER_class', 'KI67_class', 'HER2_class']

        if best_result['fold_reports']:
            first_fold_report = best_result['fold_reports'][0]

            for target_name in target_names:
                if target_name not in first_fold_report:
                    continue

                current_target_report = first_fold_report[target_name]

                rows = []
                # Filtra solo le classi '0' e '1' per evitare le medie macro nel dettaglio
                classes = [c for c in ['0', '1'] if c in current_target_report]

                for cls in classes:
                    rows.append([
                        f"Classe {cls}",
                        f"{current_target_report[cls]['precision']:.3f}",
                        f"{current_target_report[cls]['recall']:.3f}",
                        f"{current_target_report[cls]['f1-score']:.3f}",
                        int(current_target_report[cls]['support'])
                    ])

                print(f"  Target: {target_name}")
                print(tabulate(rows, headers=['', 'Precision', 'Recall', 'F1-score', 'Support'],
                             tablefmt='simple', colalign=('left', 'center', 'center', 'center', 'center')))
                print()
        else:
            print("  Nessun report dettagliato disponibile.")

        # Tabella Riassuntiva Finale
        summary_data.append([
            name,
            f"{best_result['mean_score']:.3f}",
            f"{best_result['std_score']:.3f}",
            best_result.get('max_depth', '-'),
            best_result.get('n_estimators', '-'),
            best_result.get('min_samples_leaf', '-')
        ])

    print("\n" + "=" * 80)
    print(" " * 25 + "CONFRONTO TRA TUTTI I DATASET")
    print("=" * 80 + "\n")

    # Ordina per F1-score crescente
    summary_data.sort(key=lambda x: float(x[1]), reverse=True)

    print(tabulate(summary_data,
                   headers=['Dataset', 'F1-score', 'Std Dev', 'Depth', 'Trees', 'Min Leaf'],
                   tablefmt='grid',
                   floatfmt=('.3f', '.3f', '.3f', '.0f', '.0f', '.0f')))

# Lettura dei file

In [16]:
start_time = time.time()


# Eseguo il training per tutti i dataset
results_per_dataset = {}
for name, file_path in datasets.items():
    results_per_dataset[name] = training(file_path, name)

# Usa la nuova funzione per stampare i risultati
print_grid_search_results(results_per_dataset)




end_time = time.time()
# Calcolo il tempo impiegato
execution_time = end_time - start_time
formatted_time = str(timedelta(seconds=int(execution_time)))

print("\n" + "=" * 80)
print(f" TEMPO TOTALE DI ESECUZIONE: {formatted_time}")
print("=" * 80 + "\n")



Inizio Grid Search (24 combinazioni) per: t2_medsam
Fitting 5 folds for each of 24 candidates, totalling 120 fits

Inizio Grid Search (24 combinazioni) per: t2_preprocessed
Fitting 5 folds for each of 24 candidates, totalling 120 fits

Inizio Grid Search (24 combinazioni) per: t2_original
Fitting 5 folds for each of 24 candidates, totalling 120 fits

Inizio Grid Search (24 combinazioni) per: medsam_dynamic
Fitting 5 folds for each of 24 candidates, totalling 120 fits

Inizio Grid Search (24 combinazioni) per: preprocessed_dynamic
Fitting 5 folds for each of 24 candidates, totalling 120 fits

Inizio Grid Search (24 combinazioni) per: original_dynamic
Fitting 5 folds for each of 24 candidates, totalling 120 fits

                         RIEPILOGO DEI MIGLIORI RISULTATI

────────────────────────────────────────────────────────────────────────────────
 Dataset: t2_medsam
────────────────────────────────────────────────────────────────────────────────

 Performance: F1-score = 0.667 ± 0.0